# Notebook 10: Geographic Data for Kepler.gl
**Luis Mitrovic Balbontín Archive**

Constructs two geographic datasets for visualization in Kepler.gl: an arc layer
mapping correspondence flows between origin and destination, and a point layer
locating Mitrovic himself at the time of each letter based on sender/recipient roles.
Coordinates are assigned manually from a curated dictionary of ~45 locations.

**Input:** `data/raw/LuisMitrovic_Cartas_FromAndToOnly.csv`
**Outputs:**
- `data/processed/mitrovic_cartas_kepler.csv` — arc layer (letters with complete coordinate pairs)
- `data/processed/mitrovic_ubicaciones_kepler.csv` — point layer (Mitrovic's locations)

In [3]:
from pathlib import Path

BASE       = Path.cwd().parent
DATA_RAW   = BASE / "data" / "raw"
DATA_PROC  = BASE / "data" / "processed"

DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"DATA_RAW:  {DATA_RAW}")
print(f"DATA_PROC: {DATA_PROC}")

DATA_RAW:  C:\Users\felip\mitrovic-archive\data\raw
DATA_PROC: C:\Users\felip\mitrovic-archive\data\processed


In [5]:
import pandas as pd

df = pd.read_csv(
    DATA_RAW / "LuisMitrovic_Cartas_FromAndToOnly.csv",
    sep=";",
    encoding="utf-8-sig"
)

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print(df.columns.tolist())
df.head(3)

Rows: 141 | Columns: 7
['Envelope', 'Filename', 'placeFrom', 'placeTo', 'To', 'From', 'Date']


,Envelope,Filename,placeFrom,placeTo,To,From,Date
0,S3,MITRO3-S3-DOC-001,NaN,NaN,NaN,NaN,Unkown
1,S4,MITRO3-S4-DOC-001,NaN,NaN,NaN,NaN,Unkown
2,S5,MITRO3-S5-DOC-001,NaN,NaN,NaN,NaN,Unkown


In [6]:
import sys
print(f"Python : {sys.version}")
print(f"pandas : {pd.__version__}")

Python : 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
pandas : 3.0.0


In [7]:
# Manual Corrections: cities without countries, multiline, semantic duplicates
place_corrections = {
    # placeFrom
    "New York": "New York, United States",
    "Weimar": "Weimar, Germany",
    "Frankfurt": "Frankfurt, Germany",
    "Guatemala": "Guatemala City, Guatemala",
    "Ridgeway\nLeatherhead\nSurrey, Great Britain": "Surrey, Great Britain",
    "Unkown": None,
    "Unknown": None,
    "SS STA LUCIA": None,

    # placeTo
    "Amberes, Belgium": "Antwerp, Belgium",
    "Lapad, Dubrovnik, Yugoslavia": "Dubrovnik, Croatia",
    "Lapad, Yugoslavia": "Dubrovnik, Croatia",
    "Providencia, Santiago, Chile": "Santiago, Chile",
}

df["placeFrom"] = df["placeFrom"].replace(place_corrections)
df["placeTo"]   = df["placeTo"].replace(place_corrections)

# Confirmar valores únicos resultantes
print("=== placeFrom ===")
print(sorted(df["placeFrom"].dropna().unique()))
print()
print("=== placeTo ===")
print(sorted(df["placeTo"].dropna().unique()))

=== placeFrom ===
['Algarrobo, Chile', 'Arlington, Massachusetts, USA', 'Athens, Greece', 'Barcelona, Spain', 'Beja, Portugal', 'Berlin, Germany', 'Boston, Massachusetts, USA', 'Buenos Aires, Argentina', 'Cambridge, Massachusetts, USA', 'Congonhas, Brasil', 'Cuzco, Peru', 'Den Haag, The Netherlands', 'Dubrovnik, Croatia', 'Dusseldorf, Germany', 'Frankfurt, Germany', 'Guatemala City, Guatemala', 'Hamburg, Germany', 'Herrsching Am Ammersee, Munchen, Germany', 'Huancayo, Peru', 'Islas Canarias, Spain', "Isola d'Ischia, Italy", 'Jena, Germany', 'Kabul, Afganistan', 'Kathmandu, Nepal', 'La Tour de Peilz, Switzerland', 'Lima, Peru', 'London, UK', 'Mollens, Switzerland', 'Moosburg, Austria', 'New York, United States', 'Ockley, Surrey, Great Britain', 'Old Lyme, Connecticut, USA', 'Paris, France', 'Portillo, Chile', 'Salt Lake City, Utah, United States', 'Santiago, Chile', 'Solothurn, Switzerland', 'Stockholm, Sweden', 'Stralsund, Germany', 'Surrey, Great Britain', 'Temuco, Chile', 'Tuderschit

In [8]:
# Extract year from Date column (format dd-mm-yyyy o "Unknown")
df["year"] = pd.to_numeric(df["Date"].str[-4:], errors="coerce")

# Asigning periods to letters
def assign_period(year):
    if pd.isna(year):
        return None
    elif 1934 <= year <= 1939:
        return "european"
    elif (1940 <= year <= 1971) or (1987 <= year <= 2006):
        return "chilean"
    elif 1972 <= year <= 1986:
        return "united_states"
    else:
        return None

df["period"] = df["year"].apply(assign_period)

# Summary
print(" Distribution per period ")
print(df["period"].value_counts(dropna=False))
print()
print(f"Whithout year: {df['year'].isna().sum()}")
print(f"No period (year out of range): {df['period'].isna().sum() - df['year'].isna().sum()}")

 Distribution per period 
period
european         66
chilean          57
united_states    12
NaN               6
Name: count, dtype: int64

Whithout year: 6
No period (year out of range): 0


In [9]:
# Manual coordinates for each unique places

coords = {
    "Algarrobo, Chile":                       (-33.3667, -71.6500),
    "Arlington, Massachusetts, USA":          (42.4154, -71.1565),
    "Athens, Greece":                         (37.9838,  23.7275),
    "Antwerp, Belgium":                       (51.2194,   4.4025),
    "Barcelona, Spain":                       (41.3851,   2.1734),
    "Beja, Portugal":                         (38.0154,  -7.8633),
    "Berlin, Germany":                        (52.5200,  13.4050),
    "Boston, Massachusetts, USA":             (42.3601, -71.0589),
    "Brookline, Massachusetts, USA":          (42.3318, -71.1212),
    "Buenos Aires, Argentina":               (-34.6037, -58.3816),
    "Cambridge, Massachusetts, USA":          (42.3736, -71.1097),
    "Congonhas, Brasil":                     (-20.5003, -43.8614),
    "Cuzco, Peru":                           (-13.5320, -71.9675),
    "Den Haag, The Netherlands":              (52.0705,   4.3007),
    "Dubrovnik, Croatia":                     (42.6507,  18.0944),
    "Dusseldorf, Germany":                    (51.2217,   6.7762),
    "Frankfurt, Germany":                     (50.1109,   8.6821),
    "Guatemala City, Guatemala":              (14.6349, -90.5069),
    "Hamburg, Germany":                       (53.5753,  10.0153),
    "Herrsching Am Ammersee, Munchen, Germany": (47.9968,  11.1733),
    "Huancayo, Peru":                        (-12.0651, -75.2049),
    "Islas Canarias, Spain":                  (28.2916, -16.6291),
    "Isola d'Ischia, Italy":                  (40.7310,  13.9000),
    "Jena, Germany":                          (50.9272,  11.5897),
    "Kabul, Afganistan":                      (34.5553,  69.2075),
    "Kathmandu, Nepal":                       (27.7172,  85.3240),
    "Kitzbühel, Austria":                     (47.4458,  12.3925),
    "La Tour de Peilz, Switzerland":          (46.4547,   6.8659),
    "Lima, Peru":                            (-12.0464, -77.0428),
    "London, UK":                             (51.5074,  -0.1278),
    "Martinsburg, West Virginia, United States": (39.4562, -77.9639),
    "Mollens, Switzerland":                   (46.3500,   7.5833),
    "Moosburg, Austria":                      (48.1167,  13.8667),
    "New York, United States":                (40.7128, -74.0060),
    "New York, USA":                          (40.7128, -74.0060),
    "Ockley, Surrey, Great Britain":          (51.1640,  -0.3730),
    "Old Lyme, Connecticut, USA":             (41.3154, -72.3357),
    "Paris, France":                          (48.8566,   2.3522),
    "Porto, Portugal":                        (41.1579,  -8.6291),
    "Portillo, Chile":                       (-32.9500, -70.1167),
    "Salt Lake City, Utah, United States":    (40.7608, -111.8910),
    "Santiago, Chile":                       (-33.4489, -70.6693),
    "Solothurn, Switzerland":                 (47.2088,   7.5360),
    "Stockholm, Sweden":                      (59.3293,  18.0686),
    "Stralsund, Germany":                     (54.3093,  13.0820),
    "Sun Valley, Idaho, United States":       (43.6971, -114.3533),
    "Surrey, Great Britain":                  (51.3148,  -0.5600),
    "Temuco, Chile":                         (-38.7359, -72.5904),
    "Tuderschitz, Austria":                   (46.6167,  14.2667),
    "Valparaíso, Chile":                     (-33.0472, -71.6127),
    "Vienna, Austria":                        (48.2082,  16.3738),
    "Warsow, Poland":                         (52.2297,  21.0122),
    "Washington, USA":                        (38.9072, -77.0369),
    "Washington, United States":              (38.9072, -77.0369),
    "Weimar, Germany":                        (50.9795,  11.3235),
    "Wuppertal, Germany":                     (51.2562,   7.1508),
    "Zoppot, Poland":                         (54.4414,  18.5601),
}

# Verify that each location from the corpus have coordinates
all_places = set(df["placeFrom"].dropna()) | set(df["placeTo"].dropna())
missing = all_places - set(coords.keys())

if missing:
    print("Places without coordinates:")
    for p in sorted(missing):
        print(f"  {repr(p)}")
else:
    print(f" Every location have coordinates ({len(all_places)} únicos)")

 Every location have coordinates (57 únicos)


In [10]:
# Assign lat/lon from origin and destination
df["lat_from"] = df["placeFrom"].map(lambda x: coords[x][0] if x in coords else None)
df["lon_from"] = df["placeFrom"].map(lambda x: coords[x][1] if x in coords else None)
df["lat_to"]   = df["placeTo"].map(lambda x: coords[x][0] if x in coords else None)
df["lon_to"]   = df["placeTo"].map(lambda x: coords[x][1] if x in coords else None)

# Verify coverage
total = len(df)
with_from = df["lat_from"].notna().sum()
with_to   = df["lat_to"].notna().sum()
both      = df[df["lat_from"].notna() & df["lat_to"].notna()].shape[0]

print(f"Total letters:                  {total}")
print(f"With coordinates origin:        {with_from}")
print(f"With coordinates destination:       {with_to}")
print(f"With complete pair from+to:      {both}")
print(f"Only origin:     {df[df['lat_from'].notna() & df['lat_to'].isna()].shape[0]}")
print(f"Without coordinates (excluded):   {df[df['lat_from'].isna() & df['lat_to'].isna()].shape[0]}")

Total letters:                  141
With coordinates origin:        131
With coordinates destination:       133
With complete pair from+to:      131
Only origin:     0
Without coordinates (excluded):   8


In [11]:
# Color palette 
period_colors = {
    "european":      "#9B59B6",
    "chilean":       "#16A085",
    "united_states": "#D4AC0D",
    None:            "#9F9F9F",
}

# Label for Kepler
period_labels = {
    "european":      "European (1934–1938)",
    "chilean":       "Chilean (1938–1970 / 1986–2008)",
    "united_states": "United States (1971–1985)",
    None:            "Unknown",
}

df["period_color"] = df["period"].map(lambda x: period_colors.get(x, "#9F9F9F"))
df["period_label"] = df["period"].map(lambda x: period_labels.get(x, "Unknown"))

# Tooltip for Kepler: "Año · De → A"
df["tooltip"] = df.apply(
    lambda r: f"{int(r['year']) if pd.notna(r['year']) else '?'} · {r['From']} → {r['To']}",
    axis=1
)

# Verification
print(df[["Envelope", "year", "period_label", "period_color", "tooltip"]].head(10).to_string())

  Envelope    year               period_label period_color                                              tooltip
0       S3     NaN                    Unknown      #9F9F9F                                        ? · nan → nan
1       S4     NaN                    Unknown      #9F9F9F                                        ? · nan → nan
2       S5     NaN                    Unknown      #9F9F9F                                        ? · nan → nan
3       S6  1976.0  United States (1971–1985)      #D4AC0D                                     1976 · nan → nan
4       S7  1976.0  United States (1971–1985)      #D4AC0D                 1976 · Luis Mitrovic → Chip Harkness
5       S8  1977.0  United States (1971–1985)      #D4AC0D                   1977 · Luis Mitrovic → Lem Bottoms
6      S10  1937.0       European (1934–1938)      #9B59B6           1937 · O. B. Y. → Luis Mitrovic Puljizević
7      S11  1937.0       European (1934–1938)      #9B59B6        1937 · Luis Arteaga → Luis Mitrovic Ba

In [12]:
# Columns needed for Kepler arc + metadata
kepler_cols = [
    "Envelope",
    "From", "To",
    "Date", "year",
    "period_label", "period_color",
    "placeFrom", "placeTo",
    "lat_from", "lon_from",
    "lat_to", "lon_to",
    "tooltip",
]

df_kepler = df[kepler_cols].copy()

# Only letters with complete pair
df_kepler = df_kepler[df_kepler["lat_from"].notna() & df_kepler["lat_to"].notna()]

# Export
df_kepler.to_csv(DATA_PROC / "mitrovic_cartas_kepler.csv", index=False, encoding="utf-8-sig")

print(f"Exported: {len(df_kepler)} letters")
print(f"File: mitrovic_cartas_kepler.csv")
print()
print(df_kepler.dtypes)

Exported: 131 letters
File: mitrovic_cartas_kepler.csv

Envelope            str
From                str
To                  str
Date                str
year            float64
period_label        str
period_color        str
placeFrom        object
placeTo          object
lat_from        float64
lon_from        float64
lat_to          float64
lon_to          float64
tooltip             str
dtype: object


## Data that implies Mitrovic's location

In [15]:
# Identify in which location Mitrovic was for each letter
# - If he is sender (From): he was at placeFrom
# - If he is recipient (To): he was at placeTo
# - Luis Mitrovic Puljizević is the father → excluded

def mitrovic_location(row):
    from_name = str(row["From"]) if pd.notna(row["From"]) else ""
    to_name   = str(row["To"])   if pd.notna(row["To"])   else ""

    is_sender   = "Mitrovic" in from_name and "Puljizević" not in from_name
    is_receiver = "Mitrovic" in to_name   and "Puljizević" not in to_name

    if is_sender:
        return row["placeFrom"], row["lat_from"], row["lon_from"]
    elif is_receiver:
        return row["placeTo"], row["lat_to"], row["lon_to"]
    else:
        return None, None, None

df[["mitrovic_place", "mitrovic_lat", "mitrovic_lon"]] = df.apply(
    mitrovic_location, axis=1, result_type="expand"
)

# Export only rows with know location
df_mitrovic = df[df["mitrovic_lat"].notna()][
    ["Envelope", "From", "To", "Date", "year", "period_label", "period_color",
     "mitrovic_place", "mitrovic_lat", "mitrovic_lon", "tooltip"]
].copy()

df_mitrovic.to_csv(DATA_PROC / "mitrovic_locations_kepler.csv", index=False, encoding="utf-8-sig")

print(f"Letters with Mitrovic location: {len(df_mitrovic)} / {len(df)}")
print()
print("=== Period distribution ===")
print(df_mitrovic["period_label"].value_counts(dropna=False))
print()
print("=== Unique locations where Mitrovic was ===")
print(sorted(df_mitrovic["mitrovic_place"].unique()))

Letters with Mitrovic location: 127 / 141

=== Period distribution ===
period_label
European (1934–1938)               61
Chilean (1938–1970 / 1986–2008)    56
United States (1971–1985)           9
Unknown                             1
Name: count, dtype: int64

=== Unique locations where Mitrovic was ===
['Antwerp, Belgium', 'Boston, Massachusetts, USA', 'Brookline, Massachusetts, USA', 'Dubrovnik, Croatia', 'Kabul, Afganistan', 'Kitzbühel, Austria', 'London, UK', 'Porto, Portugal', 'Santiago, Chile', 'Sun Valley, Idaho, United States', 'Vienna, Austria', 'Washington, USA', 'Weimar, Germany']


In [16]:
# How many letters situate Mitrovic in each place
freq = df_mitrovic["mitrovic_place"].value_counts().reset_index()
freq.columns = ["mitrovic_place", "carta_count"]

df_mitrovic = df_mitrovic.merge(freq, on="mitrovic_place", how="left")

# Reexport
df_mitrovic.to_csv(DATA_PROC / "mitrovic_locations_kepler.csv", index=False, encoding="utf-8-sig")

print(df_mitrovic[["mitrovic_place", "carta_count"]].drop_duplicates().sort_values("carta_count", ascending=False).to_string())

                       mitrovic_place  carta_count
0                     Santiago, Chile           64
3                     Weimar, Germany           21
12                    Vienna, Austria           21
30         Boston, Massachusetts, USA            5
27                 Kitzbühel, Austria            3
47                 Dubrovnik, Croatia            3
26                   Antwerp, Belgium            2
79      Brookline, Massachusetts, USA            2
61                         London, UK            2
6    Sun Valley, Idaho, United States            1
55                    Porto, Portugal            1
81                  Kabul, Afganistan            1
108                   Washington, USA            1
